# Dataset API Evaluation Notebook

This notebook uses the attached dataset inventory CSV to: 
1. Load dataset metadata from the local CSV file.
2. Fetch dataset metadata and records from the `data.wa.gov` Socrata API.
3. Generate improved dataset descriptions with selectable OpenAI-compatible models.
4. Compare model outputs using a judge prompt.

No frontend UI is required. Set tokens via environment variables or directly in the notebook.

## 1. Environment and dependencies

Make sure the following packages are available in your Python environment:
`pandas`, `httpx`, `python-dotenv`, `openai`.

If needed, install them with `pip install pandas httpx python-dotenv openai`.

In [ ]:
import os
import re
import json
import time
from typing import Any, Dict, List, Optional

import pandas as pd
import httpx
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

DATA_CSV_PATH = "FourMusketeersCapstone_DatasetsWithSolidMetadata(DataWA)_20260227 (2).csv"
BACKEND_URL = os.getenv("BACKEND_URL", "http://localhost:8000")
SOCRATA_APP_TOKEN = os.getenv("SOCRATA_APP_TOKEN", "")
SOCRATA_OAUTH_TOKEN = os.getenv("SOCRATA_OAUTH_TOKEN", "")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
OPENAI_API_BASE = os.getenv("OPENAI_API_BASE", "https://api.openai.com/v1")

print(
    f'BACKEND_URL={BACKEND_URL}
'
    f'SOCRATA_APP_TOKEN set={bool(SOCRATA_APP_TOKEN)}
'
    f'SOCRATA_OAUTH_TOKEN set={bool(SOCRATA_OAUTH_TOKEN)}
'
    f'OPENAI_API_KEY set={bool(OPENAI_API_KEY)}
'
    f'OPENAI_API_BASE={OPENAI_API_BASE}',
)

## 2. Load the dataset inventory CSV

This inventory includes dataset metadata such as `Name`, `Description`, `API Endpoint`, and dataset IDs.

In [ ]:
df = pd.read_csv(DATA_CSV_PATH, dtype=str, keep_default_na=False, na_filter=False)
df_columns = [c.strip() for c in df.columns]
df.columns = df_columns
print(f'Loaded {len(df)} dataset rows from CSV.')
df.head(10)

## 3. Socrata API helpers

These helpers fetch dataset metadata and rows from `data.wa.gov`. If you have a valid `SOCRATA_APP_TOKEN`, the requests will include it.

In [ ]:
def parse_socrata_dataset_id(endpoint: str) -> Optional[str]:
    if not endpoint or not isinstance(endpoint, str):
        return None
    match = re.search(r'/(?:resource|views)/([^/?#]+)', endpoint)
    return match.group(1) if match else None

def socrata_headers(app_token: str = SOCRATA_APP_TOKEN, oauth_token: str = SOCRATA_OAUTH_TOKEN) -> dict[str, str]:
    headers: dict[str, str] = {}
    if app_token:
        headers['X-App-Token'] = app_token
    if oauth_token:
        headers['Authorization'] = f'OAuth {oauth_token}'
    return headers

def fetch_socrata_metadata(dataset_id: str, headers: Optional[dict[str, str]] = None) -> dict[str, Any]:
    headers = headers or socrata_headers()
    url = f'https://data.wa.gov/api/views/{dataset_id}.json'
    with httpx.Client(timeout=60.0) as client:
        response = client.get(url, headers=headers)
        response.raise_for_status()
        return response.json()

def fetch_socrata_rows(dataset_id: str, headers: Optional[dict[str, str]] = None, page_size: int = 50000, max_rows: Optional[int] = None) -> list[dict[str, Any]]:
    headers = headers or socrata_headers()
    base_url = f'https://data.wa.gov/resource/{dataset_id}.json'
    rows: list[dict[str, Any]] = []
    offset = 0
    while True:
        params: dict[str, Any] = {'$limit': page_size, '$offset': offset}
        with httpx.Client(timeout=90.0) as client:
            response = client.get(base_url, headers=headers, params=params)
            response.raise_for_status()
            batch = response.json()
        if not batch:
            break
        rows.extend(batch)
        offset += len(batch)
        if max_rows is not None and len(rows) >= max_rows:
            rows = rows[:max_rows]
            break
        if len(batch) < page_size:
            break
        time.sleep(0.2)  # polite delay
    return rows

def fetch_socrata_via_backend(dataset_id: str, backend_url: str = BACKEND_URL, oauth_token: Optional[str] = None) -> dict[str, Any]:
    url = f'{backend_url.rstrip("/")}/api/socrata/import'
    payload = {'datasetId': dataset_id}
    if oauth_token:
        payload['oauthToken'] = oauth_token
    with httpx.Client(timeout=120.0) as client:
        response = client.post(url, json=payload)
        response.raise_for_status()
        return response.json()

## 4. Choose a dataset to evaluate

Select one row from the CSV inventory and fetch its Socrata metadata and sample rows.

In [ ]:
dataset_index = 0  # Change this index to select another row
selected = df.iloc[dataset_index] if 0 <= dataset_index < len(df) else None
if selected is None:
    raise ValueError('Invalid dataset index')

dataset_name = selected.get('Name') or selected.get('Dataset Name') or ''
dataset_description = selected.get('Description') or ''
dataset_api_endpoint = selected.get('API Endpoint') or selected.get('api endpoint') or ''
dataset_id = parse_socrata_dataset_id(dataset_api_endpoint)

print(f'Index: {dataset_index}')
print(f'Name: {dataset_name}')
print(f'Description: {dataset_description[:300]}')
print(f'API Endpoint: {dataset_api_endpoint}')
print(f'Parsed dataset ID: {dataset_id}')

if not dataset_id:
    raise ValueError('Could not parse dataset ID from API Endpoint')

print('Fetching Socrata metadata...')
socrata_meta = fetch_socrata_metadata(dataset_id)
print('Metadata loaded.')
print('Dataset display name:', socrata_meta.get('name'))
print('Dataset description length:', len(socrata_meta.get('description', '')))
if 'columns' in socrata_meta:
    print('Column count:', len([c for c in socrata_meta['columns'] if not str(c.get('fieldName', '')).startswith(':')]))

sample_rows = fetch_socrata_rows(dataset_id, max_rows=100)
print(f'Fetched {len(sample_rows)} row samples from Socrata.')

sample_rows[:5]

## 5. Build a model prompt from dataset metadata

We create a prompt that describes the dataset and asks the model to produce an improved title and description.

In [ ]:
def build_dataset_prompt(metadata: dict[str, Any], sample_rows: list[dict[str, Any]], objective: str = 'Write a polished dataset title and description in plain language.') -> str:
    dataset_name = metadata.get('name', 'Unknown dataset')
    description = metadata.get('description', '')
    columns = [c for c in metadata.get('columns', []) if not str(c.get('fieldName', '')).startswith(':')]
    column_summary = []
    for col in columns[:10]:
        column_summary.append(f

    sample_text = json.dumps(sample_rows[:5], indent=2, default=str)

    prompt = f'''
You are a metadata editor for an open government data portal.
Objective: {objective}

Dataset name: {dataset_name}
Current description: {description}

Columns:
{
        '
'.join(column_summary)
    }

Example rows (first 5):
{sample_text}

Please provide:
1. A concise improved dataset title.
2. A plain-language dataset description suitable for public data portal publication.
3. A short note on what is included in the dataset and who would find it useful.

Return your answer as plain text.
'''
    return prompt

dataset_prompt = build_dataset_prompt(socrata_meta, sample_rows)
print(dataset_prompt[:1000] + '...')

## 6. Generate model outputs and compare them

Choose model names and run the same prompt through each model. Then compare the outputs.

In [ ]:
MODEL_CHOICES = [
    "gpt5-mini",
    "gpt5-nano",
    "gpt-4.1",
    "gpt-4.1-mini",
]  # Edit this list to use the exact model names available to your API

if not OPENAI_API_KEY:
    raise ValueError('Set OPENAI_API_KEY in your environment or notebook before generating model outputs.')

client = OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_API_BASE)

def generate_model_output(model_name: str, prompt: str, max_tokens: int = 400) -> str:
    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_tokens,
        temperature=0.3,
    )
    choices = response.choices
    if not choices:
        return ''
    return ''.join([c.message.content or '' for c in choices])

model_outputs: dict[str, str] = {}
for model_name in MODEL_CHOICES:
    try:
        print(f'Generating with {model_name}...')
        text = generate_model_output(model_name, dataset_prompt)
        model_outputs[model_name] = text.strip()
        print(f'Completed {model_name} (length {len(text)}).')
    except Exception as exc:
        print(f'Failed {model_name}: {exc}')

model_outputs

## 7. Judge model outputs

Use a judge model to compare the generated outputs and select the best one.

In [ ]:
JUDGE_MODEL = "gpt-4.1-mini"  # Use a judge model available to you
JUDGE_CATEGORIES = ['clarity', 'completeness', 'accuracy', 'usefulness']

def extract_json_object(text: str) -> dict[str, Any]:
    decoder = json.JSONDecoder()
    start = text.find('{')
    if start < 0:
        raise ValueError('No JSON object found')
    obj, _ = decoder.raw_decode(text[start:])
    return obj

def build_judge_prompt(outputs: dict[str, str], categories: list[str]) -> str:
    content = [
        'You are a neutral judge comparing competing metadata descriptions for the same dataset.',
        'Evaluate the outputs using the criteria and return only valid JSON with no additional explanation.',
        'The JSON object should include:',
        '{',
        '  
: 
1
 | 
2
 | ... | 
,',
        '  
: 
,',
        '  
: 0.0-1.0,',
        '  
: {',
        '    
: {',
        '      
: 1-10,',
        '      
: 1-10,',
        '      
: 1-10,',
        '      
: 1-10',
        '    },',
        '    ...',
        '  }',
        '}',
        '',
        'Outputs to compare:',
    ]
    for idx, (model_name, text) in enumerate(outputs.items(), start=1):
        content.append(f'--- MODEL {idx}: {model_name} ---')
        content.append(text)
        content.append('')
    content.append('')
    content.append('Please return only a JSON object conforming to the schema above.')
    return '
'.join(content)

def judge_model_outputs(model_outputs: dict[str, str], judge_model: str, categories: list[str]) -> dict[str, Any]:
    prompt = build_judge_prompt(model_outputs, categories)
    response = client.chat.completions.create(
        model=judge_model,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.0,
        max_tokens=400,
    )
    text = ''.join([choice.message.content or '' for choice in response.choices])
    return extract_json_object(text)

judge_result = judge_model_outputs(model_outputs, JUDGE_MODEL, JUDGE_CATEGORIES)
judge_result

## 8. Review final outputs

View the generated description from the winning model and the judge scores.

In [ ]:
def display_winner(outputs: dict[str, str], judge_data: dict[str, Any]) -> None:
    winner = judge_data.get('winner', 'tie')
    if winner == 'tie':
        print('Judge result: tie')
        return
    winner_idx = int(winner) - 1
    model_names = list(outputs.keys())
    if winner_idx < 0 or winner_idx >= len(model_names):
        print('Invalid winner index in judge output:', winner)
        return
    winner_name = model_names[winner_idx]
    print(f'Winning model: {winner_name}')
    print('Winner reasoning:', judge_data.get('winnerReasoning'))
    print('Confidence:', judge_data.get('confidence'))
    print('
--- Winning output ---
')
    print(outputs[winner_name])

display_winner(model_outputs, judge_result)

print('
Judge scores by model:')
for model_name, scores in judge_result.get('modelScores', {}).items():
    print(model_name, scores)